# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/budnikovanastya42-dot/Internship_week_1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = **one content item, for one calendar month**, for the client that owns it.

Which table I use.
The daily content-performance table only (monthly partitions `month=YYYY-MM`, columns
`gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `gsc_data_available`).
`dim_content` and `dim_clients` are **not** used for this contract/
Which time window.
02.2026-04.2026

In [1]:
import glob, os
import numpy as np
import pandas as pd

DATA_DIR   = "."
DAILY_GLOB = "data_0*.parquet"

RNG_SEED        = 42
HISTORY_MONTH   = "2026-02"      # one month back, sole input to the trend feature
DECISION_MONTH  = "2026-03"      # mid-panel month; everything a feature may see
LABEL_MONTH     = "2026-04"      # outcome month
SEALED_MONTH    = "2026-06"      # must NOT be present
MIN_IMPRESSIONS = 50             # volume floor for the label population (contract answer 4)
DROP_PCT        = 0.20           # "declining" = impressions fell by more than 20%

files = sorted(glob.glob(os.path.join(DATA_DIR, DAILY_GLOB)))

# Which month does each exported file actually hold? Read one column, never the whole file.
rows = []
for f in files:
    s = pd.to_datetime(pd.read_parquet(f, columns=["report_date"])["report_date"])
    rows.append({"file": os.path.basename(f),
                 "month": s.dt.to_period("M").astype(str).iloc[0],
                 "daily_rows": len(s),
                 "first_day": s.min().date(), "last_day": s.max().date(),
                 "n_days": s.dt.date.nunique()})
    del s

inventory = pd.DataFrame(rows).sort_values("month").reset_index(drop=True)
month_file = {m: os.path.join(DATA_DIR, f) for m, f in zip(inventory.month, inventory.file)}
print(inventory.to_string(index=False))

for role, m in [("history", HISTORY_MONTH), ("decision", DECISION_MONTH), ("label", LABEL_MONTH)]:
    print(); print(f"{role:9s} month {m}: present in export -> {m in month_file}")
print(f"{'sealed':9s} month {SEALED_MONTH}: present in export -> {SEALED_MONTH in month_file}"
      "   <- must be False, that is the point of sealing it")

            file   month  daily_rows  first_day   last_day  n_days
data_0-4.parquet 2026-02     7355108 2026-02-01 2026-02-28      28
data_0-3.parquet 2026-03     9841378 2026-03-01 2026-03-31      31
data_0-5.parquet 2026-04    10424730 2026-04-01 2026-04-30      30
data_0-6.parquet 2026-05    11687376 2026-05-01 2026-05-31      31

history   month 2026-02: present in export -> True

decision  month 2026-03: present in export -> True

label     month 2026-04: present in export -> True
sealed    month 2026-06: present in export -> False   <- must be False, that is the point of sealing it


## 2. Fields: feature / label / context / excluded

population   content with impressions(2026-03) >= 50, present in 2026-04
is_declining = impressions(2026-04) < 0.80 * impressions(2026-03)
What a site owner cares about is "this page is losing
search visibility and is worth an hour of my time". What I can observe is "impressions fell". Those
are not the same thing — a page can lose impressions because seasonal demand fell, which is exactly
the wrong-call cost.
*The 50-impression floor is a decision rule, not a statistical one.* I checked what the floor does
to the base rate first (see the cell below): between 10 and 173 impressions the declining share
barely moves — 51.7% to 51.4%. So the floor does **not** fix a distorted base rate. What it does fix is a different thing: at no floor, 10.3% of the population are
pages that fell to exactly zero, most of them from a handful of impressions. Those are coin flips,
not decline. At a floor of 50 that drops to 1.0%. The floor is set at 50 because a reviewer would
not spend an hour on a page that got fewer than 50 impressions in a month — the threshold comes
from the decision, and the numbers only confirm it is not costly.
One thing I deliberately exclude: `fact_content_query_90d`/
Its window is `2026-04-02 … 2026-06-30`. That window starts *after* my decision moment
(2026-04-01) and ends inside the sealed test month. Any column taken from it would leak/

In [2]:
# BUILD — daily rows -> one row per content per month, then the label. Not a verification query.

GSC_SUM_COLS = ["gsc_impressions", "gsc_clicks", "gsc_sum_position"]

def monthly_frame(month):
    """Aggregate the daily partition for `month` up to one row per content item.

    Only three things happen here and each is deliberate:
      - null GSC values -> 0 / not-available. The counts are printed rather than assumed: 0.00% of
        content-days in the decision and label months, 1.25% in the history month. Treating those as
        "not available" is a choice, so it is stated here and measured below, not buried in a fillna.
      - sum, never mean. `gsc_avg_position` is a per-day average; averaging daily averages weights a
        day with 3 impressions the same as a day with 3000. The impression-weighted position is
        recovered later as sum_position / impressions, which is why sum_position is carried.
      - days_visible counts days, it does not average them.
    """
    d = pd.read_parquet(month_file[month],
                        columns=["client_hash_id", "content_hash_id", "report_date",
                                 "gsc_data_available", "ga4_data_available"] + GSC_SUM_COLS)
    # count the nulls BEFORE deciding what to do with them, so the decision stays auditable
    d["_gsc_null"] = d["gsc_data_available"].isna().astype("int8")
    d["_ga4_null"] = d["ga4_data_available"].isna().astype("int8")
    # NULL -> "not available". `.eq(True)` says that out loud instead of hiding it inside a fillna.
    for c in ["gsc_data_available", "ga4_data_available"]:
        d[c] = d[c].eq(True).astype("int8")
    for c in GSC_SUM_COLS:
        d[c] = d[c].fillna(0).astype("int64")
    g = d.groupby("content_hash_id").agg(
            client        = ("client_hash_id",     "first"),
            impressions   = ("gsc_impressions",    "sum"),
            clicks        = ("gsc_clicks",         "sum"),
            sum_position  = ("gsc_sum_position",   "sum"),
            days_visible  = ("gsc_data_available", "sum"),
            days_ga4      = ("ga4_data_available", "sum"),
            days_gsc_null = ("_gsc_null",          "sum"),
            days_ga4_null = ("_ga4_null",          "sum"),
            days_present  = ("report_date",        "size"))
    del d
    return g

feb = monthly_frame(HISTORY_MONTH)
mar = monthly_frame(DECISION_MONTH)
apr = monthly_frame(LABEL_MONTH)
for name, f in [(HISTORY_MONTH, feb), (DECISION_MONTH, mar), (LABEL_MONTH, apr)]:
    print(f"{name}: {len(f):>7,} content items, {f.impressions.gt(0).sum():>7,} with any impression, "
          f"GSC flag NULL on {f.days_gsc_null.sum()/f.days_present.sum()*100:5.2f}% of content-days, "
          f"GA4 flag NULL on {f.days_ga4_null.sum()/f.days_present.sum()*100:5.2f}%")

# --- the label -----------------------------------------------------------------------------
panel = (mar[mar.impressions >= MIN_IMPRESSIONS]
           .join(apr[["impressions"]].rename(columns={"impressions": "impressions_label_month"}),
                 how="inner"))
panel["is_declining"] = (panel.impressions_label_month
                         < (1 - DROP_PCT) * panel.impressions).astype(int)

print(); print(f"population (impressions_{DECISION_MONTH} >= {MIN_IMPRESSIONS}, present in {LABEL_MONTH}): "
      f"{len(panel):,} content items")
print(f"is_declining rate: {panel.is_declining.mean():.4f}   <- this is the floor Precision@50 must beat")

2026-02: 321,546 content items, 153,559 with any impression, GSC flag NULL on  1.25% of content-days, GA4 flag NULL on 56.45%
2026-03: 331,437 content items, 176,738 with any impression, GSC flag NULL on  0.00% of content-days, GA4 flag NULL on 30.67%
2026-04: 362,172 content items, 194,760 with any impression, GSC flag NULL on  0.00% of content-days, GA4 flag NULL on 21.26%

population (impressions_2026-03 >= 50, present in 2026-04): 116,114 content items
is_declining rate: 0.5184   <- this is the floor Precision@50 must beat


In [3]:
both = mar[["impressions"]].join(apr[["impressions"]].rename(columns={"impressions": "impr_label"}),
                                 how="inner")
print("floor  eligible   share of pop.   declining rate   fell to exactly zero")
for N in [1, 10, 30, 50, 100, 173]:
    e = both[both.impressions >= N]
    lab = e.impr_label < (1 - DROP_PCT) * e.impressions
    print(f"{N:>5} {len(e):>9,} {len(e)/ (both.impressions>=1).sum()*100:>13.1f}% "
          f"{lab.mean()*100:>15.1f}% {(e.impr_label==0).mean()*100:>21.1f}%")
del both

# (b) the four buckets, every field of the daily table placed, with the reason for each exclusion.
ga4_share      = mar.days_ga4.sum()      / mar.days_present.sum()   # NULL counted as "not available"
ga4_null_share = mar.days_ga4_null.sum() / mar.days_present.sum()
AI_COLS = ["sessions_ai","ai_chatgpt","ai_perplexity","ai_gemini","ai_copilot","ai_claude",
           "ai_meta","ai_other"]
GA4_COLS = ["ga4_pageviews","ga4_sessions","ga4_users","ga4_engaged_sessions",
            "ga4_total_engagement_sec","sessions_organic","sessions_direct","sessions_referral",
            "sessions_social","sessions_paid","scroll_events"]

contract = (
    [("content_hash_id", "key",      "identifies the contract row"),
     ("client_hash_id",  "context",  "owner; used for grouping and for the limitation in 4, never as a feature"),
     ("report_date",     "key",      "daily grain; collapsed into the month by monthly_frame()"),
     ("gsc_impressions", "feature + label",
      f"decision month -> feature; {LABEL_MONTH} -> label. The same column in two months is two different things"),
     ("gsc_clicks",      "feature",  "decision month only; feeds ctr"),
     ("gsc_sum_position","feature",  "decision month only; impression-weighted position = sum_position / impressions"),
     ("gsc_data_available","feature","decision month only, counted as days_visible. See limitation in 4 — this is not an access flag"),
     ("gsc_avg_position","excluded", "per-day average; averaging daily averages misweights low-traffic days. sum_position used instead"),
     ("client_has_gsc",  "excluded", "TRUE on ~100% of exported rows — it filters nothing, so calling it a filter would be false"),
     ("ga4_data_available","excluded", f"GA4 flag TRUE on {ga4_share*100:.1f}% of content-days in {DECISION_MONTH} and NULL on "
      f"{ga4_null_share*100:.1f}% — nothing to build a feature from, either way"),
     ("client_created_date / gsc_data_start (dim_clients)", "excluded", "dimension not read by this contract; no feature depends on it"),
     ("fact_content_query_90d (whole table)", "excluded",
      "window 2026-04-02..2026-06-30 is entirely after the decision moment and overlaps the sealed month")]
    + [(c, "excluded", f"GA4-derived; empty on {(1-ga4_share)*100:.1f}% of content-days") for c in GA4_COLS]
    + [(c, "excluded", f"AI-referral traffic, GA4-derived; empty on {(1-ga4_share)*100:.1f}% of content-days") for c in AI_COLS]
)
contract = pd.DataFrame(contract, columns=["field", "bucket", "why"])
print()
print(contract.bucket.value_counts().to_string())
print()
with pd.option_context("display.max_colwidth", 120, "display.width", 200):
    print(contract.head(12).to_string(index=False))
print(); print(f"... plus {len(GA4_COLS)+len(AI_COLS)} GA4/AI columns, all excluded for the same reason "
      f"(GA4 covers {ga4_share*100:.1f}% of content-days).")

floor  eligible   share of pop.   declining rate   fell to exactly zero
    1   176,737         100.0%            53.2%                  10.3%
   10   143,206          81.0%            51.7%                   2.7%
   30   125,645          71.1%            51.8%                   1.4%
   50   116,114          65.7%            51.8%                   1.0%
  100   101,441          57.4%            51.7%                   0.5%
  173    88,478          50.1%            51.4%                   0.3%

bucket
excluded           24
feature             3
key                 2
feature + label     1
context             1

                                             field          bucket                                                                                                              why
                                   content_hash_id             key                                                                                      identifies the contract row
                       

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# the grain
d = pd.read_parquet(month_file[DECISION_MONTH],
                    columns=["client_hash_id", "content_hash_id", "report_date"])

n_rows   = len(d)
n_cd     = len(d.drop_duplicates(["content_hash_id", "report_date"]))
n_ccd    = len(d.drop_duplicates(["client_hash_id", "content_hash_id", "report_date"]))

print(f"raw daily rows in {DECISION_MONTH}              : {n_rows:,}")
print(f"unique (content_hash_id, report_date)      : {n_cd:,}")
print(f"unique (client, content, report_date)      : {n_ccd:,}")
print(f"-> warehouse grain is (content, day)       : {n_rows == n_cd}")
print(f"-> client adds nothing to the key          : {n_cd == n_ccd}   (one content belongs to one client)")

print(); print(f"after monthly_frame(): rows {len(mar):,}, unique content_hash_id {mar.index.nunique():,}"
      f"  -> contract row is unique: {len(mar) == mar.index.nunique()}")
print(f"days per content in the month: min {mar.days_present.min()}, "
      f"median {int(mar.days_present.median())}, max {mar.days_present.max()}"
      "   <- not every content has a row every day, so 'one row per content-month' is an aggregation, not a reshape")

raw daily rows in 2026-03              : 9,841,378
unique (content_hash_id, report_date)      : 9,841,378
unique (client, content, report_date)      : 9,841,378
-> warehouse grain is (content, day)       : True
-> client adds nothing to the key          : True   (one content belongs to one client)

after monthly_frame(): rows 331,437, unique content_hash_id 331,437  -> contract row is unique: True
days per content in the month: min 1, median 31, max 31   <- not every content has a row every day, so 'one row per content-month' is an aggregation, not a reshape


In [ ]:
# the slice
s = pd.to_datetime(d["report_date"])
print(f"slice = {DECISION_MONTH} partition")
print(f"  rows            : {len(d):,}")
print(f"  date span       : {s.min().date()} .. {s.max().date()}  ({s.dt.date.nunique()} distinct days)")
print(f"  content items   : {d.content_hash_id.nunique():,}")
print(f"  clients         : {d.client_hash_id.nunique():,}")
print(f"  span sits entirely before the decision moment 2026-04-01: {s.max().date() < pd.Timestamp('2026-04-01').date()}")
print(); print(f"contract population after the volume floor and the {LABEL_MONTH} join: {len(panel):,} rows, "
      f"{panel.client.nunique()} clients")
del d, s

In [5]:
# availability, filtered with IS TRUE
a = pd.read_parquet(month_file[DECISION_MONTH], columns=["gsc_data_available", "gsc_impressions"])
a["gsc_data_available"] = a["gsc_data_available"].fillna(False)

total    = len(a)
survived = int((a.gsc_data_available == True).sum())          # IS TRUE
print(f"rows before filter                         : {total:,}")
print(f"rows surviving `gsc_data_available IS TRUE`: {survived:,}  ({survived/total*100:.1f}%)")

# and what that flag actually is — kept for section 4
avail_check = pd.crosstab(a.gsc_data_available, a.gsc_impressions > 0)
avail_check.index.name = "gsc_data_available"; avail_check.columns.name = "gsc_impressions > 0"
print(); print(avail_check.to_string())
identity = (avail_check.values[0, 1] == 0) and (avail_check.values[1, 0] == 0)
print(); print(f"`gsc_data_available` is exactly `gsc_impressions > 0`, no exceptions in {total:,} rows: {identity}")
del a

rows before filter                         : 9,841,378
rows surviving `gsc_data_available IS TRUE`: 3,611,061  (36.7%)

gsc_impressions > 0    False    True 
gsc_data_available                   
False                6230317        0
True                       0  3611061

`gsc_data_available` is exactly `gsc_impressions > 0`, no exceptions in 9,841,378 rows: True


## 4. Data limits
 gsc_data_available is not an availability flag.

Query 3 checked it against every row of the decision month, and the cross-tab has two empty cells and
no exceptions: the column is exactly `gsc_impressions > 0`, renamed. Three consequences, in order of
how much they hurt:

1.The data cannot distinguish "this page did not rank" from "we have no data for this page."

2. `client_has_gsc` does not rescue the situation**, because it is `TRUE` on essentially every
   exported row. It filters nothing.

Three further limits, stated because they are real, not to pad the section:

- **The panel grows inside the window.** Content that first appears in the label month has no
  decision-month row and is silently outside the population — correctly, but it means the contract
  describes surviving-and-established content, not all content.
- **The client mix is very uneven.** The counts are printed below. A ranking fitted on this panel is
  disproportionately fitted to the largest client.
- **One label month is one month of weather.** April 2026 carries whatever algorithm change and
  seasonality that month happened to contain. Nothing here separates "this page is decaying" from
  "April was bad for this topic", and no amount of care with the features fixes that — it needs more
  label months, which is a later notebook, not this one.

In [6]:
print("`gsc_data_available` vs `gsc_impressions > 0`, decision month, every row:")
print(avail_check.to_string())
print(); print(f"off-diagonal cells (disagreements): "
      f"{int(avail_check.values[0,1]) + int(avail_check.values[1,0])}")

per_client = panel.client.value_counts()
print(); print(f"client mix in the contract population ({len(panel):,} rows, {per_client.size} clients):")
print(f"  median rows per client : {int(per_client.median()):,}")
print(f"  largest client         : {per_client.iloc[0]:,} rows "
      f"({per_client.iloc[0]/len(panel)*100:.1f}% of the population)")
print(f"  top 3 clients together : {per_client.head(3).sum()/len(panel)*100:.1f}% of the population")

print(); print(f"content present in {LABEL_MONTH} but not in {DECISION_MONTH} (outside the population by design): "
      f"{len(apr.index.difference(mar.index)):,}")
print(f"content present in {DECISION_MONTH} but gone in {LABEL_MONTH}: "
      f"{len(mar.index.difference(apr.index)):,}  <- attrition is not a threat to this label")

`gsc_data_available` vs `gsc_impressions > 0`, decision month, every row:
gsc_impressions > 0    False    True 
gsc_data_available                   
False                6230317        0
True                       0  3611061

off-diagonal cells (disagreements): 0

client mix in the contract population (116,114 rows, 44 clients):
  median rows per client : 368
  largest client         : 23,659 rows (20.4% of the population)
  top 3 clients together : 48.7% of the population

content present in 2026-04 but not in 2026-03 (outside the population by design): 30,736
content present in 2026-03 but gone in 2026-04: 1  <- attrition is not a threat to this label


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown
thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.